<a href="https://colab.research.google.com/github/ethnicgarbage/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/Jun_Seo_Lee_2026_09_18_%E2%80%94_Pandas_Challenge_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [ ]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [ ]:
# TODO
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()

print(f'total revenue: ${total_revenue:.2f}')
print(f'total units: {len(df)}')

total revenue: $8520.00
total units: 400


##Q1 - Writeup

To find the revenue count, I multiplied the quantity by the price as that would get me the revenue for that category. To find the total revenue, I would have to sum the revenue from all the categories.

In terms of the total units, I would have to find the length of the transaction list.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [ ]:
# TODO
category_revenue = (df.groupby('category', as_index=False)
.agg(revenue=('revenue', 'sum'))
.sort_values('revenue', ascending=False))

category_revenue['Share of Total (%)'] = 100 * category_revenue['revenue'] / total_revenue

category_revenue = category_revenue.round(2)
category_revenue

,category,revenue,Share of Total (%)
1,Food,4293.0,50.39
2,Merch,1771.5,20.79
0,Drink,1554.0,18.24
3,RainGear,901.5,10.58


##Q2 - Writeup
As I knew there was already a revenue calculation done, I could then group the revenue by category using the groupby function. The revenue values within each category were summed using the groupby() and sum() functions. Using the sort_values function, I sort the revenue by category from highest to lowest through ascending=False).

To find the percentage share of revenue, I had to divide the category revenue by the total and multiply by 100. Once this was put into a dataframe, I ensured to round to two decimal places for legibility and added titles that would help a user ascertain the aim of the each column.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [ ]:
# TODO
vendor_average = (df.groupby('vendor_id').agg(
    avg_order_revenue=('revenue', 'mean'),
    order_count=('revenue', 'count')
))
vendor_average = vendor_average.sort_values(by='avg_order_revenue', ascending=False)
vendor_average.round(2)

print(vendor_average)

           avg_order_revenue  order_count
vendor_id                                
V-01               22.595745           94
V-18               21.750000          108
V-05               20.580645           93
V-10               20.314286          105


##Q3 - Writeup
The data was grouped by the vendor_id and .agg was used to calculate the mean for revenue and order count for each vendor. The results were then rounded to two decimal places and sorted by the average revenue in descending order for legibility.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [ ]:
# TODO
merch_revenue = df[df['category'] == 'Merch']['revenue'].sum()
merch_share = 100 * merch_revenue / total_revenue

print(f'Merch revenue share: {merch_share:.1f}%')

Merch revenue share: 20.8%


##Q4 - Writeup

To find the share of revenue that comes from merch, I had to first find the total revenue generated specifically for merch. To do this, I created a new calculation that would look at summing revenue only if the category matched merch.

Following this, I found the share of merch by dividing merch_revenue by the total_revenue that was made earlier and multiplying by 100 to get it to a percentage.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [ ]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
baseline_rows = len(df)
baseline_revenue = df['revenue'].sum()

clean_vendors = vendor_names.drop_duplicates()
assert clean_vendors['vendor_id'].is_unique, 'vendor_id must be unique to join safely'

joined = df.merge(clean_vendors, on='vendor_id', how='left', indicator=True)

print('rows:', len(joined), '(expected', baseline_rows, ')')
print('revenue:', joined['revenue'].sum(), '(expected', baseline_revenue, ')')

assert len(joined) == baseline_rows
assert joined['revenue'].sum() == baseline_revenue

missing = joined[joined['_merge'] == 'left_only']
print(f'Missing vendor IDs: {len(missing)} | '
      f'revenue at stake: ${missing["revenue"].sum():.2f}')

joined['vendor_name'] = joined['vendor_name'].fillna('Unknown vendor')
joined = joined.drop(columns='_merge')

joined[['vendor_id', 'vendor_name', 'category', 'qty', 'price', 'revenue']]

rows: 400 (expected 400 )
revenue: 8520.0 (expected 8520.0 )
Missing vendor IDs: 108 | revenue at stake: $2349.00


,vendor_id,vendor_name,category,qty,price,revenue
0,V-10,Cav Merch North,Drink,2,24.0,48.0
1,V-18,Unknown vendor,RainGear,1,12.0,12.0
2,V-18,Unknown vendor,Drink,3,4.5,13.5
3,V-10,Cav Merch North,Food,2,12.0,24.0
4,V-18,Unknown vendor,Drink,3,7.5,22.5
...,...,...,...,...,...,...
395,V-18,Unknown vendor,Merch,1,12.0,12.0
396,V-01,Hoos Burgers,Merch,2,24.0,48.0
397,V-10,Cav Merch North,Food,3,7.5,22.5
398,V-18,Unknown vendor,Merch,2,24.0,48.0


**The unmatched vendor, and what I did about it:** _..._
As V-18 was missing from vendor_names. I decided to retain the 108 orders as it is still extremely important to record the data around the unknown vendor so that it's category, quantity, price, and revenue information was retained.

By keeping these unmatched vendors under an 'unknown vendor' label, and merging left and ensuring that they were still joined into the total list, I was able to retain this information.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [ ]:
# TODO
lab_4_pivot = pd.pivot_table(
    joined,
    values='revenue',
    index='vendor_name',
    columns='category',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
)
lab_4_pivot

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown vendor,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


##Q6 - Writeup

To create a pivot table, I Googled how to create a pivot table and used the information found here: https://pandas.pydata.org/docs/reference/api/pandas.pivot_table.html to complete the task.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [ ]:
by_category = category_revenue

assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.


**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.


As food is a primary driver for revenue with 50.39% of the total revenue share, it is the strongest category by a wide margin. As a result, the vendors should look to prioritize selling food at the next game. Things such as drinks and merch were able to be sold at $1554 and $1771.5 respectively. However, rain gear only managed to produce $901.5 worth of revenue, for just 10.58% so should be avoided unless there is inclement weather that necessitates rain gear.

I believe that revenue by category (Q2) is the most unreliable in terms of providing information that would help determine a reliable analysis. Revenue by category here does not actually provide information on the reasoning behind why certain categories are strong/weak. For example, rain gear only produced $901.5 worth of revenue but it does not provide information on if the weather was inclement and if that points towards why rain gear sales were so weak. A similar argument could be made about the strength of the other categories. If the timing of the game matched the timing necessary for strong food and drink sales, then that could be the primary driver. Revenue by category would be more trustworthy if there was averages and more information on the situation around the games.